# Sideband Analysis & SBI Pipeline Demo

This notebook demonstrates the full axion search analysis chain:
1. **Load channel images** (dirty for now, cleaned once available)
2. **Sideband background subtraction** — remove frequency-smooth emission
3. **Residual inspection** — what remains after background removal
4. **Dummy NS template bank** — placeholder for Sam Witte's real templates
5. **SBI interface sketch** — how Josh Foster's CASCAL pipeline would connect

Uses subband 30 (~1156-1166 MHz) as a test case — mid-band, low RFI.

In [ ]:
import os
import sys
import glob
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm
from astropy.io import fits
from astropy.wcs import WCS
import warnings
warnings.filterwarnings('ignore')

# Add project scripts to path
PROJECT_DIR = '/global/scratch/projects/pc_heptheory/jbenabou/NS_megaproject/MeerKAT_data/meerkat_reduction_project'
sys.path.insert(0, os.path.join(PROJECT_DIR, 'scripts'))

# Import sideband analysis functions from phase7
from phase7_axion_search import (
    load_fits_image, save_fits_image, radec_to_pixel, extract_cutout,
    get_sideband_channels, build_background_model, sideband_subtract,
    measure_local_rms, measure_peak_flux, compute_significance,
    freq_to_global_channel, global_channel_to_subband, channel_to_freq_mhz,
    find_channel_fits, load_rfi_flags, load_ns_template_bank,
    NSTarget, IMAGES_DIR, CHANS_PER_SUBBAND
)

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
print("Imports OK")

## 1. Load and visualize channel images

Load a few channels from subband 30 to see what the data looks like.

In [ ]:
# Load a few channels from subband 30
SUBBAND = 30
USE_CLEANED = False  # Switch to True once cleaned images are available

# Pick channels spread across the subband
test_channels = [10, 100, 200, 300]
images = {}
headers = {}

for ch in test_channels:
    fpath = find_channel_fits(SUBBAND, ch, cleaned=USE_CLEANED)
    if fpath and os.path.exists(fpath):
        img, hdr = load_fits_image(fpath)
        images[ch] = img
        headers[ch] = hdr
        freq = channel_to_freq_mhz(SUBBAND, ch)
        print(f"  Channel {ch}: {freq:.3f} MHz, shape={img.shape}, "
              f"peak={np.nanmax(img)*1e3:.1f} mJy, rms={np.nanstd(img)*1e3:.1f} mJy")

# Plot the 4 channels
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for ax, ch in zip(axes.flat, test_channels):
    if ch in images:
        freq = channel_to_freq_mhz(SUBBAND, ch)
        im = ax.imshow(images[ch] * 1e3, origin='lower', cmap='inferno',
                       vmin=-10, vmax=300)
        ax.set_title(f'Ch {ch} ({freq:.1f} MHz)')
        plt.colorbar(im, ax=ax, label='mJy/beam', shrink=0.8)
fig.suptitle(f'Subband {SUBBAND} — Raw Channel Images (dirty)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'plots', 'sideband_demo_raw_channels.png'), dpi=150)
plt.show()

## 2. Sideband Background Subtraction

For a given "signal" channel, build a background model from nearby channels (the sidebands), then subtract. The residual should remove frequency-smooth emission (continuum, sidelobes) and retain only frequency-dependent features (RFI, noise, potential axion signals).

In [ ]:
# Perform sideband subtraction on channel 100 of subband 30
signal_chan = 100
n_sideband = 50   # channels on each side
n_guard = 5       # channels to skip around signal

# Load RFI flags
rfi_flags = load_rfi_flags()
print(f"RFI flags loaded: {len(rfi_flags)} flagged channels")

# Get sideband channels
sideband_chans = get_sideband_channels(SUBBAND, signal_chan, n_sideband, n_guard, rfi_flags)
print(f"Sideband channels: {len(sideband_chans)} (from {2*(n_sideband - n_guard)} possible)")

# Build background model (median of sideband channels)
print("Building background model (this loads ~90 FITS files)...")
background, n_loaded = build_background_model(sideband_chans, cleaned=USE_CLEANED, method='median')
print(f"Background model built from {n_loaded} channels")

# Load signal channel
signal_path = find_channel_fits(SUBBAND, signal_chan, cleaned=USE_CLEANED)
signal_image, signal_header = load_fits_image(signal_path)
signal_freq = channel_to_freq_mhz(SUBBAND, signal_chan)

# Subtract
residual = sideband_subtract(signal_image, background)

print(f"\nSignal channel {signal_chan} ({signal_freq:.3f} MHz):")
print(f"  Signal peak:     {np.nanmax(signal_image)*1e3:.3f} mJy")
print(f"  Background peak: {np.nanmax(background)*1e3:.3f} mJy")
print(f"  Residual peak:   {np.nanmax(residual)*1e3:.3f} mJy")
print(f"  Residual RMS:    {np.nanstd(residual)*1e3:.3f} mJy")
print(f"  Residual min:    {np.nanmin(residual)*1e3:.3f} mJy")

In [ ]:
# Plot: Signal, Background, Residual side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Signal channel
im0 = axes[0].imshow(signal_image * 1e3, origin='lower', cmap='inferno', vmin=-10, vmax=300)
axes[0].set_title(f'Signal Ch {signal_chan}\n({signal_freq:.1f} MHz)')
plt.colorbar(im0, ax=axes[0], label='mJy/beam', shrink=0.8)

# Background model
im1 = axes[1].imshow(background * 1e3, origin='lower', cmap='inferno', vmin=-10, vmax=300)
axes[1].set_title(f'Background Model\n(median of {n_loaded} sidebands)')
plt.colorbar(im1, ax=axes[1], label='mJy/beam', shrink=0.8)

# Residual (symmetric colorscale)
resid_vmax = 3 * np.nanstd(residual) * 1e3
im2 = axes[2].imshow(residual * 1e3, origin='lower', cmap='RdBu_r',
                      vmin=-resid_vmax, vmax=resid_vmax)
axes[2].set_title(f'Residual (signal - background)\nRMS={np.nanstd(residual)*1e3:.2f} mJy')
plt.colorbar(im2, ax=axes[2], label='mJy/beam', shrink=0.8)

fig.suptitle('Sideband Background Subtraction', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'plots', 'sideband_demo_subtraction.png'), dpi=150)
plt.show()

## 3. Residual Statistics & Noise Properties

Check whether the residual is Gaussian noise (as expected if sideband subtraction removes all systematic structure).

In [ ]:
# Residual histogram vs Gaussian
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

resid_flat = residual.flatten() * 1e3  # mJy
resid_valid = resid_flat[np.isfinite(resid_flat)]

# Histogram
ax = axes[0]
ax.hist(resid_valid, bins=100, density=True, alpha=0.7, color='steelblue', label='Residual')
# Overlay Gaussian
from scipy.stats import norm
mu, sigma = np.mean(resid_valid), np.std(resid_valid)
x = np.linspace(mu - 5*sigma, mu + 5*sigma, 200)
ax.plot(x, norm.pdf(x, mu, sigma), 'r-', lw=2, label=f'Gaussian ($\\mu$={mu:.2f}, $\\sigma$={sigma:.2f} mJy)')
ax.set_xlabel('Residual flux (mJy/beam)')
ax.set_ylabel('Probability density')
ax.set_title('Residual Histogram')
ax.legend()

# QQ plot
from scipy.stats import probplot
ax = axes[1]
probplot(resid_valid, plot=ax)
ax.set_title('Q-Q Plot (residual vs Gaussian)')
ax.get_lines()[0].set_markersize(1)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'plots', 'sideband_demo_residual_stats.png'), dpi=150)
plt.show()

print(f"Residual statistics:")
print(f"  Mean:     {mu:.4f} mJy")
print(f"  Std:      {sigma:.4f} mJy")
print(f"  Skewness: {float(np.mean(((resid_valid - mu)/sigma)**3)):.4f}")
print(f"  Kurtosis: {float(np.mean(((resid_valid - mu)/sigma)**4)) - 3:.4f}  (0 = Gaussian)")

## 4. Sideband Subtraction Across Multiple Channels

Run sideband subtraction on many channels to see how residual RMS varies with frequency. This reveals:
- Channels with excess variance (potential RFI or real spectral features)
- Whether the residual noise floor is uniform across the band

In [ ]:
# Run sideband subtraction across channels 10-370 (skip edges)
# Sample every 5th channel to keep it fast
sample_channels = list(range(10, 370, 5))
resid_rms_by_chan = []
resid_peak_by_chan = []
freqs_mhz = []

print(f"Processing {len(sample_channels)} channels...")
for i, ch in enumerate(sample_channels):
    freq = channel_to_freq_mhz(SUBBAND, ch)
    freqs_mhz.append(freq)
    
    # Get sideband channels
    sb_chans = get_sideband_channels(SUBBAND, ch, n_sideband, n_guard, rfi_flags)
    
    # Build background
    bg, n_bg = build_background_model(sb_chans, cleaned=USE_CLEANED, method='median')
    
    # Load signal
    fpath = find_channel_fits(SUBBAND, ch, cleaned=USE_CLEANED)
    if fpath is None or bg is None:
        resid_rms_by_chan.append(np.nan)
        resid_peak_by_chan.append(np.nan)
        continue
    
    sig_img, _ = load_fits_image(fpath)
    resid = sideband_subtract(sig_img, bg)
    
    resid_rms_by_chan.append(np.nanstd(resid) * 1e3)
    resid_peak_by_chan.append(np.nanmax(np.abs(resid)) * 1e3)
    
    if (i+1) % 20 == 0:
        print(f"  {i+1}/{len(sample_channels)} done")

print("Done!")

# Plot residual RMS vs frequency
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(freqs_mhz, resid_rms_by_chan, 'b.-', markersize=3, lw=0.5)
axes[0].set_ylabel('Residual RMS (mJy/beam)')
axes[0].set_title(f'Subband {SUBBAND}: Sideband Residual Statistics vs Frequency')
axes[0].axhline(np.nanmedian(resid_rms_by_chan), color='r', ls='--', 
                label=f'Median RMS = {np.nanmedian(resid_rms_by_chan):.2f} mJy')
axes[0].legend()

axes[1].plot(freqs_mhz, resid_peak_by_chan, 'r.-', markersize=3, lw=0.5)
axes[1].set_ylabel('|Peak residual| (mJy/beam)')
axes[1].set_xlabel('Frequency (MHz)')
axes[1].axhline(np.nanmedian(resid_peak_by_chan), color='b', ls='--',
                label=f'Median peak = {np.nanmedian(resid_peak_by_chan):.2f} mJy')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'plots', 'sideband_demo_rms_vs_freq.png'), dpi=150)
plt.show()

## 5. Dummy NS Template Bank & Target Search Demo

Use the dummy NS template bank to demonstrate the full search pipeline. These targets are near the Galactic Center — once we have Sam Witte's real template bank, we swap in the real positions and predicted axion frequencies.

In [ ]:
# Load dummy NS template bank
targets = load_ns_template_bank()
print(f"Template bank: {len(targets)} targets\n")
for t in targets:
    print(f"  {t.name}: RA={t.ra_deg:.4f}, Dec={t.dec_deg:.4f}, "
          f"freq={t.predicted_freq_mhz:.1f} MHz, extent={t.signal_extent_arcsec}\"")

# For demo, pick a target whose frequency falls in our subband 30 range
# Subband 30: ~1156-1166 MHz. None of the dummy targets are in this range,
# so let's create a fake test target at a channel in subband 30
demo_target = NSTarget(
    name="DEMO_NS_test",
    ra_deg=266.417,       # Near Sgr A*
    dec_deg=-29.008,
    predicted_freq_mhz=channel_to_freq_mhz(SUBBAND, 150),  # channel 150
    signal_extent_arcsec=10.0,
)
print(f"\nDemo target: freq={demo_target.predicted_freq_mhz:.3f} MHz "
      f"(subband {SUBBAND}, channel 150)")

In [ ]:
# Run the full search pipeline on the demo target
from phase7_axion_search import search_single_target

result = search_single_target(
    demo_target, rfi_flags,
    n_sideband=50, n_guard=5, threshold_sigma=5.0,
    bg_method='median', use_cleaned=USE_CLEANED, save_residuals=True,
)

if result:
    print(f"\n=== Search Result ===")
    print(f"  Target:        {result.ns_name}")
    print(f"  Frequency:     {result.freq_mhz:.3f} MHz")
    print(f"  Peak residual: {result.peak_residual_flux*1e3:.4f} mJy")
    print(f"  Local RMS:     {result.local_rms*1e3:.4f} mJy")
    print(f"  Significance:  {result.significance:.2f} sigma")
    print(f"  Candidate:     {result.is_candidate}")
    print(f"  Sideband used: {result.n_sideband_used}")

## 6. Injected Signal Test

Inject a fake axion signal into a channel image to verify the sideband pipeline can detect it. The signal is a 2D Gaussian (representing a spatially resolved axion conversion region) added to one channel but not the sidebands.

In [ ]:
def make_gaussian_signal(shape, x_center, y_center, sigma_pix, amplitude_jy):
    """Create a 2D Gaussian signal at a given position."""
    ny, nx = shape
    yy, xx = np.mgrid[:ny, :nx]
    r2 = (xx - x_center)**2 + (yy - y_center)**2
    return amplitude_jy * np.exp(-r2 / (2 * sigma_pix**2))

# Inject into channel 200 of subband 30
inject_chan = 200
inject_fpath = find_channel_fits(SUBBAND, inject_chan, cleaned=USE_CLEANED)
inject_img, inject_hdr = load_fits_image(inject_fpath)

# Injection parameters
inject_x, inject_y = 300, 250  # arbitrary position
inject_sigma_pix = 5            # ~10 arcsec FWHM
inject_amplitudes = [0.5e-3, 1e-3, 2e-3, 5e-3, 10e-3]  # Jy/beam

# Build background from sidebands (no injection in sidebands)
sb_chans = get_sideband_channels(SUBBAND, inject_chan, n_sideband, n_guard, rfi_flags)
bg, n_bg = build_background_model(sb_chans, cleaned=USE_CLEANED, method='median')

print(f"Injection test: channel {inject_chan}, position ({inject_x}, {inject_y})")
print(f"Background from {n_bg} sidebands\n")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes_flat = axes.flatten()

for i, amp in enumerate(inject_amplitudes):
    signal = make_gaussian_signal(inject_img.shape, inject_x, inject_y, inject_sigma_pix, amp)
    injected = inject_img + signal
    resid = sideband_subtract(injected, bg)
    
    # Measure detection
    peak = measure_peak_flux(resid, inject_x, inject_y, aperture_radius_pix=15)
    rms = measure_local_rms(resid, inject_x, inject_y, exclude_radius_pix=20)
    snr = compute_significance(peak, rms)
    
    print(f"  Injected {amp*1e3:.1f} mJy: peak={peak*1e3:.3f} mJy, "
          f"RMS={rms*1e3:.3f} mJy, SNR={snr:.1f}")
    
    # Plot cutout around injection site
    ax = axes_flat[i]
    cutout = extract_cutout(resid * 1e3, inject_x, inject_y, 40)
    vmax = max(3*rms*1e3, np.nanmax(cutout)*0.8)
    im = ax.imshow(cutout, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_title(f'Injected: {amp*1e3:.1f} mJy\nSNR={snr:.1f}')
    plt.colorbar(im, ax=ax, label='mJy', shrink=0.8)

# Last panel: no injection (null test)
resid_null = sideband_subtract(inject_img, bg)
cutout_null = extract_cutout(resid_null * 1e3, inject_x, inject_y, 40)
im = axes_flat[5].imshow(cutout_null, origin='lower', cmap='RdBu_r', 
                          vmin=-vmax, vmax=vmax)
axes_flat[5].set_title('No injection (null)')
plt.colorbar(im, ax=axes_flat[5], label='mJy', shrink=0.8)

fig.suptitle('Signal Injection Recovery Test', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'plots', 'sideband_demo_injection.png'), dpi=150)
plt.show()

## 7. SBI Interface Sketch

How Josh Foster's CASCAL pipeline (AxionLinePop) would connect to our data.
The SBI code expects 3D numpy datacubes (spatial x spatial x frequency).
Here we sketch the data extraction step that would feed into the neural network.

In [ ]:
def extract_datacube(subband_idx, center_chan, x_center, y_center,
                     spatial_half=8, n_freq_channels=16, cleaned=False):
    """
    Extract a 3D datacube (spatial x spatial x freq) centered on a target.
    
    This is the interface between our FITS images and Josh Foster's SBI code.
    The SBI network expects cubes of shape (N_spatial, N_spatial, N_freq).
    
    Parameters
    ----------
    subband_idx : int
    center_chan : int -- central channel (where axion signal is expected)
    x_center, y_center : int -- pixel position of NS target
    spatial_half : int -- half-width of spatial cutout (default 8 -> 16x16)
    n_freq_channels : int -- number of frequency channels (default 16)
    cleaned : bool -- use cleaned images
    
    Returns
    -------
    cube : ndarray of shape (2*spatial_half, 2*spatial_half, n_freq_channels)
    freq_mhz : ndarray of channel frequencies
    """
    half_freq = n_freq_channels // 2
    chan_start = center_chan - half_freq
    chan_end = center_chan + half_freq
    
    cube_slices = []
    freq_list = []
    
    for ch in range(chan_start, chan_end):
        # Handle subband boundaries
        if ch < 0 or ch >= CHANS_PER_SUBBAND:
            cube_slices.append(np.zeros((2*spatial_half, 2*spatial_half)))
            freq_list.append(np.nan)
            continue
            
        fpath = find_channel_fits(subband_idx, ch, cleaned=cleaned)
        if fpath is None:
            cube_slices.append(np.zeros((2*spatial_half, 2*spatial_half)))
            freq_list.append(np.nan)
            continue
        
        img, _ = load_fits_image(fpath)
        cutout = extract_cutout(img, x_center, y_center, spatial_half)
        
        # Pad if near edge
        if cutout.shape != (2*spatial_half+1, 2*spatial_half+1):
            padded = np.zeros((2*spatial_half+1, 2*spatial_half+1))
            padded[:cutout.shape[0], :cutout.shape[1]] = cutout
            cutout = padded
        
        # Trim to exact size
        cube_slices.append(cutout[:2*spatial_half, :2*spatial_half])
        freq_list.append(channel_to_freq_mhz(subband_idx, ch))
    
    return np.stack(cube_slices, axis=-1), np.array(freq_list)


# Extract a demo datacube at the injection position
print("Extracting 16x16x16 datacube...")
cube, freqs = extract_datacube(SUBBAND, 200, 300, 250, 
                                spatial_half=8, n_freq_channels=16, cleaned=USE_CLEANED)
print(f"Cube shape: {cube.shape}")
print(f"Freq range: {freqs[0]:.3f} - {freqs[-1]:.3f} MHz")
print(f"Cube min/max: {np.nanmin(cube)*1e3:.2f} / {np.nanmax(cube)*1e3:.2f} mJy")

# Visualize the datacube
fig, axes = plt.subplots(2, 8, figsize=(20, 5))
for i, ax in enumerate(axes.flat):
    if i < cube.shape[2]:
        im = ax.imshow(cube[:,:,i] * 1e3, origin='lower', cmap='inferno', 
                       vmin=-5, vmax=300)
        ax.set_title(f'{freqs[i]:.1f}', fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])

fig.suptitle('16x16x16 Datacube (each panel = 1 freq channel)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'plots', 'sideband_demo_datacube.png'), dpi=150)
plt.show()

print("\n--- SBI Integration Notes ---")
print("Josh Foster's AxionLinePop expects:")
print(f"  Input shape: {cube.shape} -- matches the 16x16x16 default")
print("  Framework: TensorFlow/Keras with CASCAL loss")
print("  Next steps:")
print("    1. Get Sam Witte's real NS template bank")
print("    2. Generate training data with realistic MeerKAT noise model")
print("    3. Train CASCAL classifier on (signal+noise) vs (noise-only) cubes")
print("    4. Run inference on all target positions in real data")

## 8. Summary & Next Steps

**What this notebook demonstrated:**
- Sideband background subtraction removes frequency-smooth emission (continuum + sidelobes)
- Residual noise properties (Gaussianity, RMS vs frequency)
- Signal injection & recovery at various amplitudes
- Data extraction for SBI interface (16x16x16 datacubes)

**Remaining to do:**
1. **Cleaning**: Get tclean working with correct thresholds, then re-run with cleaned images (`USE_CLEANED=True`)
2. **Sam Witte's template bank**: Replace dummy NS targets with real predicted axion frequencies & positions
3. **SBI training**: Generate training cubes with realistic MeerKAT noise, train CASCAL network
4. **Full search**: Run sideband analysis + SBI across all 32,768 channels and all NS targets
5. **Sensitivity curve**: Map residual RMS to upper limits on axion-photon coupling $g_{a\gamma}$

## 9. Cleaning Demo: Dirty Beam Convolution & Deconvolution

Pedagogical demonstration of what tclean does. We simulate:
1. A model sky with point sources
2. Convolve with a synthesized "dirty beam" (PSF) — this is what the interferometer measures
3. Add thermal noise → "dirty image"
4. Run a simple Hogbom CLEAN algorithm to deconvolve
5. Compare dirty vs cleaned images

This helps verify that our cleaning thresholds and approach are correct.

In [ ]:
from scipy.signal import fftconvolve

def make_meerkat_psf(size=512, cell_arcsec=2.0, freq_mhz=1160.0):
    """
    Simulate a MeerKAT-like dirty beam (PSF).
    MeerKAT at L-band: ~7" synthesized beam (FWHM) at 1.4 GHz,
    scales as lambda/D. At 1160 MHz: ~8.4" FWHM.
    The PSF has sidelobes from the array's uv-coverage.
    """
    beam_fwhm_arcsec = 7.0 * (1400.0 / freq_mhz)  # Scale with frequency
    beam_sigma_pix = beam_fwhm_arcsec / (2.355 * cell_arcsec)
    
    center = size // 2
    yy, xx = np.mgrid[:size, :size]
    r2 = (xx - center)**2 + (yy - center)**2
    
    # Main beam (Gaussian)
    main_beam = np.exp(-r2 / (2 * beam_sigma_pix**2))
    
    # Add sidelobes: concentric rings with decreasing amplitude
    # This mimics the Airy-like sidelobe pattern of an interferometer
    r = np.sqrt(r2)
    sidelobes = np.zeros_like(main_beam)
    for ring_r, ring_amp in [(12, 0.15), (20, -0.08), (30, 0.05), (42, -0.03)]:
        ring_width = 3.0
        sidelobes += ring_amp * np.exp(-(r - ring_r)**2 / (2 * ring_width**2))
    
    psf = main_beam + sidelobes
    psf /= psf.max()  # Normalize peak to 1
    return psf, beam_fwhm_arcsec


def make_model_sky(size=512, n_sources=5):
    """Create a model sky with point sources of varying brightness."""
    np.random.seed(42)
    sky = np.zeros((size, size))
    
    # Central bright source (like Sgr A*)
    cx, cy = size//2, size//2
    sky[cy, cx] = 1.0  # 1 Jy
    
    # Additional sources at random positions
    positions = []
    fluxes = [1.0]
    for _ in range(n_sources - 1):
        x = np.random.randint(size//4, 3*size//4)
        y = np.random.randint(size//4, 3*size//4)
        flux = 10**(np.random.uniform(-2, -0.3))  # 10 mJy to 500 mJy
        sky[y, x] = flux
        positions.append((x, y))
        fluxes.append(flux)
    
    return sky, fluxes


def hogbom_clean(dirty, psf, niter=5000, gain=0.1, threshold=0.0):
    """
    Simple Hogbom CLEAN deconvolution.
    
    1. Find the peak in the residual image
    2. Subtract a scaled copy of the PSF centered at the peak
    3. Record the clean component
    4. Repeat until threshold or niter
    """
    residual = dirty.copy()
    model = np.zeros_like(dirty)
    size = dirty.shape[0]
    psf_center = psf.shape[0] // 2
    
    peaks = []
    
    for i in range(niter):
        # Find peak in residual
        peak_idx = np.unravel_index(np.argmax(np.abs(residual)), residual.shape)
        peak_val = residual[peak_idx]
        
        if abs(peak_val) < threshold:
            print(f"  CLEAN stopped at iter {i}: peak {abs(peak_val)*1e3:.3f} mJy < threshold {threshold*1e3:.3f} mJy")
            break
        
        peaks.append(abs(peak_val))
        
        # Record clean component
        model[peak_idx] += gain * peak_val
        
        # Subtract scaled, shifted PSF from residual
        py, px = peak_idx
        # Compute overlap region
        y_lo = max(0, py - psf_center)
        y_hi = min(size, py + psf_center + 1)
        x_lo = max(0, px - psf_center)
        x_hi = min(size, px + psf_center + 1)
        
        psf_y_lo = psf_center - (py - y_lo)
        psf_y_hi = psf_center + (y_hi - py)
        psf_x_lo = psf_center - (px - x_lo)
        psf_x_hi = psf_center + (x_hi - px)
        
        residual[y_lo:y_hi, x_lo:x_hi] -= gain * peak_val * psf[psf_y_lo:psf_y_hi, psf_x_lo:psf_x_hi]
    
    if i == niter - 1:
        print(f"  CLEAN reached niter={niter}, final peak={abs(peak_val)*1e3:.3f} mJy")
    
    return model, residual, peaks


# 1. Create model sky
model_sky, source_fluxes = make_model_sky(512, n_sources=8)
print(f"Model sky: {len(source_fluxes)} sources")
for i, f in enumerate(source_fluxes):
    print(f"  Source {i}: {f*1e3:.1f} mJy")

# 2. Make PSF
psf, beam_fwhm = make_meerkat_psf(512, 2.0, 1160.0)
print(f"\nPSF FWHM: {beam_fwhm:.1f} arcsec ({beam_fwhm/2.0:.1f} pixels)")

# 3. Convolve to make dirty image
dirty = fftconvolve(model_sky, psf, mode='same')

# 4. Add noise
noise_rms = 0.001  # 1 mJy thermal noise
noise = np.random.normal(0, noise_rms, dirty.shape)
dirty_noisy = dirty + noise

print(f"\nDirty image: peak={np.max(dirty_noisy)*1e3:.1f} mJy, RMS={np.std(dirty_noisy)*1e3:.1f} mJy")
print(f"Noise RMS: {noise_rms*1e3:.1f} mJy")
print(f"Peak SNR: {np.max(dirty_noisy)/noise_rms:.0f}")

In [ ]:
# 5. Run CLEAN with different thresholds to show the effect
thresholds = {
    'Too high (3x full-image RMS)': 3 * np.std(dirty_noisy),   # BAD: includes source flux in "RMS"
    'Correct (3x noise)': 3 * noise_rms,                        # GOOD: uses actual thermal noise
    'Deep (1x noise)': 1 * noise_rms,                           # Very deep clean
}

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# Row 0: input images
axes[0,0].imshow(model_sky * 1e3, origin='lower', cmap='inferno', vmin=0, vmax=50)
axes[0,0].set_title('True Sky (model)')
axes[0,1].imshow(psf[200:312, 200:312], origin='lower', cmap='RdBu_r')
axes[0,1].set_title(f'Dirty Beam (PSF)\nFWHM={beam_fwhm:.1f}"')
axes[0,2].imshow(dirty_noisy * 1e3, origin='lower', cmap='inferno', vmin=-5, vmax=300)
axes[0,2].set_title(f'Dirty Image\n(peak={np.max(dirty_noisy)*1e3:.0f} mJy)')
axes[0,3].axis('off')

# Row 1: cleaned images with different thresholds
for col, (label, thr) in enumerate(thresholds.items()):
    print(f"\nCLEAN with threshold = {thr*1e3:.2f} mJy ({label}):")
    model_clean, resid_clean, peaks_clean = hogbom_clean(dirty_noisy, psf, niter=10000, 
                                                          gain=0.1, threshold=thr)
    
    # Restore: convolve model with clean beam (Gaussian) + add residual
    clean_beam_sigma = beam_fwhm / (2.355 * 2.0)  # pixels
    yy, xx = np.mgrid[:512, :512]
    clean_beam = np.exp(-((xx-256)**2 + (yy-256)**2) / (2*clean_beam_sigma**2))
    clean_beam /= clean_beam.sum()
    restored = fftconvolve(model_clean, clean_beam, mode='same') + resid_clean
    
    ax = axes[1, col]
    ax.imshow(restored * 1e3, origin='lower', cmap='inferno', vmin=-5, vmax=300)
    ax.set_title(f'{label}\nthr={thr*1e3:.1f} mJy, iters={len(peaks_clean)}')
    
    print(f"  Clean components: {np.count_nonzero(model_clean)}")
    print(f"  Model flux sum: {model_clean.sum()*1e3:.1f} mJy")
    print(f"  Residual RMS: {np.std(resid_clean)*1e3:.2f} mJy")

# The "bad" case: what happens with wrong threshold
axes[1, 3].axis('off')
axes[1, 3].text(0.1, 0.5, 
    'Key lesson:\n\n'
    'Using full-image RMS as\n'
    '"noise" is WRONG when\n'
    'the image contains bright\n'
    'sources + sidelobes.\n\n'
    'The threshold must be\n'
    'based on actual thermal\n'
    'noise (off-source region),\n'
    'NOT imstat RMS over\n'
    'the whole image.',
    fontsize=12, transform=axes[1,3].transAxes, verticalalignment='center',
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

fig.suptitle('CLEAN Deconvolution: Effect of Threshold Choice', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'plots', 'sideband_demo_clean_thresholds.png'), dpi=150)
plt.show()

In [ ]:
# 6. CLEAN convergence: peak residual vs iteration
fig, ax = plt.subplots(figsize=(10, 5))

for label, thr in thresholds.items():
    _, _, peaks = hogbom_clean(dirty_noisy, psf, niter=10000, gain=0.1, threshold=thr)
    ax.semilogy(range(len(peaks)), [p*1e3 for p in peaks], label=f'{label} (thr={thr*1e3:.1f} mJy)')

ax.axhline(noise_rms * 1e3, color='gray', ls=':', label=f'Thermal noise ({noise_rms*1e3:.1f} mJy)')
ax.set_xlabel('CLEAN Iteration')
ax.set_ylabel('|Peak Residual| (mJy)')
ax.set_title('CLEAN Convergence: How threshold affects stopping')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'plots', 'sideband_demo_clean_convergence.png'), dpi=150)
plt.show()